## 03 LLM Product Enrichment

Transform raw product metadata into structured, marketing-ready features using rule-based logic and LLM-based enrichment.

Created attributes:

- style
- occasion
- material_hint
- target_audience
- selling_points
- marketing_keywords
- copy_angle

Output:

- articles_enriched.csv (product catalog with enriched marketing features)


#### Load Product Catalog


In [1]:
import os
import json
import pandas as pd

In [2]:
PROCESSED_DIR = "../data/processed/hm"

articles = pd.read_csv(os.path.join(PROCESSED_DIR, "articles.csv"))

articles.head()

,article_id,product_code,product_name,product_type_no,product_type,product_group,graphical_appearance_no,graphical_appearance,colour_group_code,color_group,...,section_no,section,garment_group_no,garment_group,description,image_path,image_exists,product_purchase_count,unique_customer_count,avg_selling_price
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,../data/raw/hm/images/010/0108775015.jpg,True,175.0,172.0,0.008139
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,../data/raw/hm/images/010/0108775044.jpg,True,116.0,116.0,0.008196
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,../data/raw/hm/images/010/0108775051.jpg,True,2.0,2.0,0.004559
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",../data/raw/hm/images/011/0110065001.jpg,True,19.0,19.0,0.020756
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",../data/raw/hm/images/011/0110065002.jpg,True,11.0,11.0,0.016932


#### Create Product Context


In [5]:
product_context_cols = [
    "article_id",
    "product_name",
    "product_type",
    "product_group",
    "color_group",
    "index_group",
    "garment_group",
    "description"
]

products = articles[product_context_cols].copy()

In [ ]:
# Create product context text
products["product_context"] = (
    "Product name: " + products["product_name"].fillna("") + "\n"
    + "Type: " + products["product_type"].fillna("") + "\n"
    + "Group: " + products["product_group"].fillna("") + "\n"
    + "Color: " + products["color_group"].fillna("") + "\n"
    + "Index group: " + products["index_group"].fillna("") + "\n"
    + "Garment group: " + products["garment_group"].fillna("") + "\n"
    + "Description: " + products["description"].fillna("")
)

# Preview product context
products[["article_id", "product_context"]].head()

,article_id,product_context
0,108775015,Product name: Strap top\nType: Vest top\nGroup...
1,108775044,Product name: Strap top\nType: Vest top\nGroup...
2,108775051,Product name: Strap top (1)\nType: Vest top\nG...
3,110065001,Product name: OP T-shirt (Idro)\nType: Bra\nGr...
4,110065002,Product name: OP T-shirt (Idro)\nType: Bra\nGr...


#### Rule-Based Enrichment Baseline


In [7]:
# Rule-based style inference
def infer_style(row):
    text = " ".join([
        str(row.get("product_name", "")),
        str(row.get("product_type", "")),
        str(row.get("garment_group", "")),
        str(row.get("detail_desc", ""))
    ]).lower()

    if any(word in text for word in ["sport", "training", "running", "gym"]):
        return "sporty"
    elif any(word in text for word in ["dress", "blouse", "shirt", "tailored"]):
        return "elegant"
    elif any(word in text for word in ["hoodie", "sweatshirt", "tee", "t-shirt"]):
        return "casual"
    elif any(word in text for word in ["denim", "jacket", "cargo"]):
        return "streetwear"
    else:
        return "everyday"

In [8]:
# Rule-based occasion inference
def infer_occasion(row):
    text = " ".join([
        str(row.get("product_name", "")),
        str(row.get("product_type", "")),
        str(row.get("garment_group", "")),
        str(row.get("detail_desc", ""))
    ]).lower()

    if any(word in text for word in ["sport", "training", "running", "gym"]):
        return "workout"
    elif any(word in text for word in ["dress", "blouse", "shirt"]):
        return "work_or_outing"
    elif any(word in text for word in ["swim", "bikini"]):
        return "vacation"
    else:
        return "daily_wear"

In [9]:
# Apply rule-based enrichment
products["style"] = products.apply(infer_style, axis=1)
products["occasion"] = products.apply(infer_occasion, axis=1)

products[["article_id", "product_name", "style", "occasion"]].head()

,article_id,product_name,style,occasion
0,108775015,Strap top,everyday,daily_wear
1,108775044,Strap top,everyday,daily_wear
2,108775051,Strap top (1),everyday,daily_wear
3,110065001,OP T-shirt (Idro),elegant,work_or_outing
4,110065002,OP T-shirt (Idro),elegant,work_or_outing


#### LLM Prompt Design


In [10]:
def build_enrichment_prompt(product_context):
    return f"""
        You are a retail marketing assistant.

        Given the product information below, return a JSON object with:
        - style
        - occasion
        - material_hint
        - target_audience
        - selling_points
        - marketing_keywords
        - copy_angle

        Product information:
        {product_context}

        Return only valid JSON.
"""

#### LLM Enrichment Function


In [11]:
# This is a placeholder version of the LLM.
# Instead of calling an API, it creates fake-but-structured enrichment so that we can:
# build and test the pipeline first without paying for or waiting on LLM calls.
# We can replace this function with a real DeepSeek / OpenAI / Hugging Face call later.
def mock_llm_enrich(row):
    return {
        "style": row["style"],
        "occasion": row["occasion"],
        "material_hint": "unknown",
        "target_audience": row["index_group"],
        "selling_points": [
            f"Easy to style {row['product_type']}",
            f"Versatile for {row['occasion']}"
        ],
        "marketing_keywords": [
            row["style"],
            row["occasion"],
            row["color_group"]
        ],
        "copy_angle": f"{row['style']} and {row['occasion']} focused"
    }

In [18]:
# Generate enrichment records
enriched_records = []

for _, row in products.iterrows():
    enrichment = mock_llm_enrich(row)
    enriched_records.append({
        "article_id": row["article_id"],
        **enrichment
    })

enrichment_df = pd.DataFrame(enriched_records)

enrichment_df.head()

,article_id,style,occasion,material_hint,target_audience,selling_points,marketing_keywords,copy_angle
0,108775015,everyday,daily_wear,unknown,Ladieswear,"[Easy to style Vest top, Versatile for daily_w...","[everyday, daily_wear, Black]",everyday and daily_wear focused
1,108775044,everyday,daily_wear,unknown,Ladieswear,"[Easy to style Vest top, Versatile for daily_w...","[everyday, daily_wear, White]",everyday and daily_wear focused
2,108775051,everyday,daily_wear,unknown,Ladieswear,"[Easy to style Vest top, Versatile for daily_w...","[everyday, daily_wear, Off White]",everyday and daily_wear focused
3,110065001,elegant,work_or_outing,unknown,Ladieswear,"[Easy to style Bra, Versatile for work_or_outing]","[elegant, work_or_outing, Black]",elegant and work_or_outing focused
4,110065002,elegant,work_or_outing,unknown,Ladieswear,"[Easy to style Bra, Versatile for work_or_outing]","[elegant, work_or_outing, White]",elegant and work_or_outing focused


#### Merge Enriched Attributes


In [19]:
# This joins the new marketing attributes back to the full product catalog.
# Products outside the sample will have missing values for those new fields.
articles_enriched = articles.merge(
    enrichment_df,
    on="article_id",
    how="left"
)

articles_enriched.head()

,article_id,product_code,product_name,product_type_no,product_type,product_group,graphical_appearance_no,graphical_appearance,colour_group_code,color_group,...,product_purchase_count,unique_customer_count,avg_selling_price,style,occasion,material_hint,target_audience,selling_points,marketing_keywords,copy_angle
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,175.0,172.0,0.008139,everyday,daily_wear,unknown,Ladieswear,"[Easy to style Vest top, Versatile for daily_w...","[everyday, daily_wear, Black]",everyday and daily_wear focused
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,116.0,116.0,0.008196,everyday,daily_wear,unknown,Ladieswear,"[Easy to style Vest top, Versatile for daily_w...","[everyday, daily_wear, White]",everyday and daily_wear focused
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,2.0,2.0,0.004559,everyday,daily_wear,unknown,Ladieswear,"[Easy to style Vest top, Versatile for daily_w...","[everyday, daily_wear, Off White]",everyday and daily_wear focused
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,19.0,19.0,0.020756,elegant,work_or_outing,unknown,Ladieswear,"[Easy to style Bra, Versatile for work_or_outing]","[elegant, work_or_outing, Black]",elegant and work_or_outing focused
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,11.0,11.0,0.016932,elegant,work_or_outing,unknown,Ladieswear,"[Easy to style Bra, Versatile for work_or_outing]","[elegant, work_or_outing, White]",elegant and work_or_outing focused


#### Save Output


In [21]:
output_path = os.path.join(PROCESSED_DIR, "articles_enriched.csv")

articles_enriched.to_csv(output_path, index=False)

print(output_path)
print(articles_enriched.shape)

../data/processed/hm/articles_enriched.csv
(105542, 37)


#### Output Check


In [22]:
articles_enriched[
    [
        "article_id",
        "product_name",
        "product_type",
        "color_group",
        "style",
        "occasion",
        "target_audience",
        "copy_angle"
    ]
].head()

,article_id,product_name,product_type,color_group,style,occasion,target_audience,copy_angle
0,108775015,Strap top,Vest top,Black,everyday,daily_wear,Ladieswear,everyday and daily_wear focused
1,108775044,Strap top,Vest top,White,everyday,daily_wear,Ladieswear,everyday and daily_wear focused
2,108775051,Strap top (1),Vest top,Off White,everyday,daily_wear,Ladieswear,everyday and daily_wear focused
3,110065001,OP T-shirt (Idro),Bra,Black,elegant,work_or_outing,Ladieswear,elegant and work_or_outing focused
4,110065002,OP T-shirt (Idro),Bra,White,elegant,work_or_outing,Ladieswear,elegant and work_or_outing focused
